# 1. 数据读取

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, accuracy_score
)

df = pd.read_csv('data/processed_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

if df['cancelled'].dtype != bool:
    df['cancelled'] = (
        df['cancelled'].astype(str).str.strip().str.lower()
        .isin(['true', '1', 'yes', 'y'])
    )

df = df.dropna(subset=['CustomerID', 'InvoiceNo', 'InvoiceDate']).copy()
df['CustomerID'] = df['CustomerID'].astype(str)
df = df[df['Quantity'] > 0].copy()

print('有效交易明细行数：', len(df))
print('客户数：', df['CustomerID'].nunique())
print('日期范围：', df['InvoiceDate'].min(), '至', df['InvoiceDate'].max())
display(df.head())

# 2. 客户特征和数据集划分

使用参考日期之前的有效交易构造特征，预测之后 90 天是否复购：有有效购买记为 `future_purchase=1`，否则为 0。复购为正类。
参考 RFM.ipynb 按客户直接聚合。两种特征共享训练集、验证集和测试集。

In [ ]:
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
reference_date = analysis_date - pd.Timedelta(days=90)

valid = df.loc[~df['cancelled']]
history = valid.loc[valid['InvoiceDate'] <= reference_date]
future = valid.loc[valid['InvoiceDate'] > reference_date]

data = history.groupby('CustomerID').agg(
    LastPurchaseDate=('InvoiceDate', 'max'),
    OrderCount=('InvoiceNo', 'nunique'),
    TotalSpend=('TotalPrice', 'sum'),
    TotalQuantity=('Quantity', 'sum')
).reset_index()
data['Recency'] = (reference_date - data.pop('LastPurchaseDate')).dt.days
data['AvgOrderValue'] = data['TotalSpend'] / data['OrderCount']
data['AvgItemsPerOrder'] = data['TotalQuantity'] / data['OrderCount']
data['future_purchase'] = data['CustomerID'].isin(future['CustomerID']).astype(int)
data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

train_df, temp_df = train_test_split(
    data, test_size=0.30, random_state=42, stratify=data['future_purchase']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['future_purchase']
)

print('客户样本量：', len(data))
print('训练集：', len(train_df))
print('验证集：', len(val_df))
print('测试集：', len(test_df))
display(data.head())

# 3. 基于直接聚合特征预测用户未来90天复购情况

In [ ]:
target = 'future_purchase'

# 两种特征共用训练和评估步骤，避免复制整段模型代码。
def train_and_evaluate(feature_cols):
    X_train = train_df[feature_cols]
    y_train = train_df[target].astype(int)
    X_val = val_df[feature_cols]
    y_val = val_df[target].astype(int)

    model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=42,
        tree_method='hist'
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    results = {}
    for name, frame in [('验证集', val_df), ('测试集', test_df)]:
        y_true = frame[target].astype(int)
        y_score = model.predict_proba(frame[feature_cols])[:, 1]
        y_pred = (y_score >= 0.5).astype(int)
        results[name] = {
            'Accuracy': accuracy_score(y_true, y_pred),
            'AUC': roc_auc_score(y_true, y_score),
            'PR-AUC': average_precision_score(y_true, y_score),
            'F1': f1_score(y_true, y_pred, zero_division=0),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'Confusion Matrix': confusion_matrix(y_true, y_pred, labels=[0, 1])
        }

    print('特征列：', feature_cols)
    print('最佳迭代轮数：', model.best_iteration)
    for name, metrics in results.items():
        print(f'\n{name}')
        for metric in ['Accuracy', 'AUC', 'PR-AUC', 'F1', 'Precision', 'Recall']:
            print(f'{metric}: {metrics[metric]:.4f}')
        print('混淆矩阵（行=真实，列=预测；顺序：未复购、复购）：')
        print(metrics['Confusion Matrix'])
    return model, results

In [ ]:
direct_features = [
    'TotalSpend', 'OrderCount', 'AvgOrderValue',
    'TotalQuantity', 'AvgItemsPerOrder'
]
model_direct, metrics_direct = train_and_evaluate(direct_features)

# 4. 基于RFM特征预测用户未来90天复购情况

在直接聚合特征上增加 Recency 和 R/F/M 评分，衡量其增量价值。
评分方向与 RFM.ipynb 一致；为避免泄漏，仅在训练集计算分位点。相同数值不按客户排名拆分，重复分位点合并后实际评分档数可能少于五档。

In [ ]:
# R/F/M 分别对应 Recency、OrderCount（Frequency）、TotalSpend（Monetary）。
# 仅从训练集获取五分位点，再应用于验证集和测试集。
score_sources = {
    'R_Score': 'Recency',
    'F_Score': 'OrderCount',
    'M_Score': 'TotalSpend'
}
for score, source in score_sources.items():
    cuts = np.unique(train_df[source].quantile([0.2, 0.4, 0.6, 0.8]))
    for frame in [train_df, val_df, test_df]:
        values = np.searchsorted(cuts, frame[source], side='right') + 1
        frame[score] = 6 - values if score == 'R_Score' else values

rfm_features = direct_features + ['Recency', 'R_Score', 'F_Score', 'M_Score']
model_rfm, metrics_rfm = train_and_evaluate(rfm_features)

# 5. 两个模型效果的对比

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

metric_names = ['Accuracy', 'AUC', 'PR-AUC', 'F1', 'Precision', 'Recall']
comparison_tables = {}
for split in ['验证集', '测试集']:
    comparison_tables[split] = pd.DataFrame({
        '直接聚合特征': [metrics_direct[split][name] for name in metric_names],
        'RFM特征': [metrics_rfm[split][name] for name in metric_names]
    }, index=metric_names)
    print(split)
    display(comparison_tables[split].round(4))

comparison = comparison_tables['测试集']
display((comparison['RFM特征'] - comparison['直接聚合特征'])
        .rename('测试集指标变化').round(4))

ax = comparison.plot(
    kind='bar',
    figsize=(10, 5),
    ylim=(0, 1),
    color=['#4C78A8', '#F58518'],
    alpha=0.9
)
ax.set_title('测试集未来90天复购预测指标对比')
ax.set_ylabel('分数')
ax.set_xlabel('评价指标')
ax.legend(title='特征')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

结合 AUC、PR-AUC 和固定 0.5 阈值下的分类指标评价两种特征。指标变化表示增加 Recency 和评分后的整体效果，单次划分不能证明统计显著性。